# 02 Feature Construction and Labeling

Edited by Wang Hongjie


## 0. Function Definition and Basic Configuration

In [1]:
from pathlib import Path
from decimal import Decimal
import numpy as np
import pandas as pd
import hashlib
import json
from datetime import datetime
from IPython.display import display

# Run from SP500; all notebook helpers are local.
ROOT = Path.cwd().resolve()
if not (ROOT / "01_data_preprocessing.ipynb").is_file():
    raise FileNotFoundError("Run this notebook from the SP500 directory.")

### Path and Feature Config

In [2]:
DATA_DIR = ROOT / "data"

# external market data directory
MARKET_DIR = DATA_DIR / "market"

CLEAN_FILE = DATA_DIR / "processed" / "sp500_clean.csv"

FEATURE_DIR = DATA_DIR / "features"

RAW_FILE = DATA_DIR / "raw" / "sp500_daily_1996-11-13_to_2026-11-12_raw.json"

# basic market characteristics
ORIGINAL = [
    "return_1d", "return_5d", "return_20d", "volatility_20d",
    "intraday_return", "intraday_range",
]

# external market characteristics
MARKET = [
    "vix_close", "vix_change", "treasury_2y_change_bp",
    "dxy_change_pct", "wti_change_usd",
]

MARKET_FILES = [
    MARKET_DIR / "vix_history.csv",
    *[MARKET_DIR / name for name in (
        "treasury_archive.csv", "treasury_2025.csv", "treasury_example.csv",
        "dxy.json", "wti.json",
    )],
]

>- VIX **Panic index** - Market sentiment
>- **Treasury yields** - Interest rate environment
>- DXY **Dollar Index** - Exchange rate environment
>- WTI **crude oil price** - Energy market

### Table, image and file saving

In [3]:
def save_csv(path, frame, index=True):
    frame.to_csv(path, index=index, encoding="utf-8-sig", index_label="signal_date" if index else None)

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

### Load data and build targets

In [4]:
# build predictive targets
def add_targets(bars, features):
    """
    at the close of day t, 
    predict to buy at the opening on day t+1 and sell at the opening on day t+2
    """
    dates = bars.index.to_series()
    mills = (bars["open"] * 1000).round().astype("Int64")
    entry, exit_ = mills.shift(-1), mills.shift(-2)
    samples = features.assign(
        trade_date=dates.shift(-1), settle_date=dates.shift(-2),
        target_return=bars["open"].shift(-2) / bars["open"].shift(-1) - 1,
        target_class=(500 * (exit_ - entry) > entry).astype("Int8"),
    )
    return samples.replace([np.inf, -np.inf], np.nan).dropna(subset=[*ORIGINAL, "target_return", "target_class"])

# read external market closing price
def _market_close(filename, symbol):
    chart = json.loads((MARKET_DIR / filename).read_text(encoding="utf-8"))["chart"]["result"][0]
    if chart["meta"]["symbol"] != symbol:
        raise ValueError(f"{filename} has an incorrect ticker.")
    dates = pd.to_datetime(chart["timestamp"], unit="s", utc=True)
    dates = dates.tz_convert("America/New_York").tz_localize(None).normalize()
    values = pd.Series(chart["indicators"]["quote"][0]["close"], index=dates, dtype=float).dropna().sort_index()
    if not values.index.is_unique:
        raise ValueError(f"{filename} has duplicate dates.")
    return values

# build external market characteristics
# Calculate external market characteristics, align transaction dates, and prevent future information leakage
def load_market_features(dates):
    # read VIX panic index
    raw = pd.read_csv(MARKET_FILES[0], usecols=["DATE", "CLOSE"])
    vix = raw.set_index(pd.to_datetime(raw["DATE"], format="%m/%d/%Y"))["CLOSE"].sort_index()
    # read the yield of US two-year treasury bond bonds in different time periods
    yield_parts = []
    for filename, fmt in (("treasury_archive.csv", "%m/%d/%y"),
                          ("treasury_2025.csv", "%m/%d/%Y"), ("treasury_example.csv", "%m/%d/%Y")):
        raw = pd.read_csv(MARKET_DIR / filename, usecols=["Date", "2 Yr"])
        yield_parts.append(raw.set_index(pd.to_datetime(raw["Date"], format=fmt))["2 Yr"])
    treasury = pd.concat(yield_parts, verify_integrity=True).dropna().sort_index()
    # read USD Index and WTI Crude Oil Prices
    dxy, wti = _market_close("dxy.json", "DX-Y.NYB"), _market_close("wti.json", "CL=F")
    # calculate 5 external market characteristics
    series = [vix, vix.diff(), treasury.diff() * 100, dxy.pct_change(fill_method=None) * 100, wti.diff()]
    features, audit = pd.DataFrame(index=dates), []
    for name, values in zip(MARKET, series, strict=True):
        delay = np.ones(len(values), dtype=int)
        if name == "treasury_2y_change_bp":
            delay[values.index < "2016-10-11"] = 7
        available = values.index + pd.to_timedelta(delay, unit="D")
        aligned = pd.DataFrame({"value": values.to_numpy(), "source_date": values.index,
                                "available_date": available}, index=available)
        aligned = aligned.loc[~aligned.index.duplicated(keep="last")].sort_index().dropna(subset=["value"])
        aligned = aligned.reindex(dates, method="ffill", tolerance=pd.Timedelta(days=7))
        valid = aligned["value"].notna()
        if not (aligned.loc[valid, "available_date"] <= dates[valid]).all():
            raise AssertionError("Cross-market feature uses data after the signal date.")
        features[name] = aligned["value"]
        audit.append(aligned.assign(feature=name, signal_date=dates))
    return features, pd.concat(audit, ignore_index=True)

**How to prevent data leakage from external data?**

1. External market data is delayed by one natural day by default.
2. Treasury bond data before October 11, 2016 was delayed for seven calendar days.
3. Missing data will only be filled with historical available data, not future data.
4. Check if the available data date is later than the predicted date.

### Chronological split

> It is required that the selling date of the training sample **must be earlier than** the starting trading date of the validation set. Otherwise, the training set labels may contain future price information during the validation set period.

In [5]:
def chronological_split(samples, ratios=(0.8, 0.1, 0.1)):
    """Split valid samples by date; drop targets crossing boundaries."""
    if len(ratios) != 3 or not np.isclose(sum(ratios), 1) or min(ratios) <= 0:
        raise ValueError("Provide three positive ratios summing to one.")
    val_size, test_size = (round(len(samples) * x) for x in ratios[1:])
    train_size = len(samples) - val_size - test_size
    if min(train_size, val_size, test_size) < 2:
        raise ValueError("Too few samples in a split.")
    validation_start = samples.iloc[train_size]["trade_date"]
    test_start = samples.iloc[train_size + val_size]["trade_date"]
    # exclude samples where transactions cross the boundary of the division
    splits = {
        "train": samples.loc[samples.settle_date < validation_start].copy(),
        "val": samples.loc[(samples.trade_date >= validation_start) & (samples.settle_date < test_start)].copy(),
        "test": samples.loc[samples.trade_date >= test_start].copy(),
    }
    assert all(len(frame) > 0 for frame in splits.values())    
    # check if the settlement date of the previous data set is later than the prediction date of the later data set
    assert splits["train"].settle_date.max() <= splits["val"].index.min()
    assert splits["val"].settle_date.max() <= splits["test"].index.min()
    # check for duplicates
    all_dates = pd.Index(np.concatenate([f.index.to_numpy() for f in splits.values()]))
    assert all_dates.is_unique, "Duplicate signal dates across splits."
    summary = pd.DataFrame([
        {"split": name, "samples": len(frame), "first_signal": frame.index.min(),
         "last_signal": frame.index.max(), "first_entry": frame.trade_date.min(),
         "last_settlement": frame.settle_date.max()}
        for name, frame in splits.items()
    ])
    return splits, summary

## 1. Read the cleaned original partition

read and merge S&P 500 market data, and verify data quality again before feature engineering.

In [6]:
# 1. Load prices and validate inputs
# verify data quality again before feature engineering

DATA_DIR = ROOT / "data" / "splits"
OUTPUT_DIR = FEATURE_DIR

FILES = {split: DATA_DIR / f"sp500_{split}.csv" for split in ("train", "val", "test")}
required = {"date", "open", "high", "low", "close", "volume"}
parts = []

print("=" * 65)
print("[1/5] Loading S&P 500 prices")
for split, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path, parse_dates=["date"], float_precision="round_trip")
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing)}")
    if df.empty:
        raise ValueError(f"{path.name} is empty")
    df = df.sort_values("date")
    parts.append(df)
    print(f"  {split:5s}: {len(df):5,d} rows | {df['date'].min():%Y-%m-%d} ~ {df['date'].max():%Y-%m-%d}")

split_starts = [part["date"].min() for part in parts[1:]]
daily = pd.concat(parts, ignore_index=True).set_index("date").sort_index()
daily.index.name = "signal_date"

assert daily.index.notna().all() and daily.index.is_unique, "Dates contain missing or duplicate values"
assert daily.index.is_monotonic_increasing, "Dates are not sorted"
assert len(daily) >= 3, "Insufficient data"
prices = daily[["open", "high", "low", "close"]].to_numpy(dtype=float)  # Validate OHLC before building features
assert (np.isfinite(prices) & (prices > 0)).all(), "OHLC contains invalid or nonpositive prices"
assert (daily["high"] >= daily[["open", "close", "low"]].max(axis=1)).all(), "High price is inconsistent with OHLC"
assert (daily["low"] <= daily[["open", "close", "high"]].min(axis=1)).all(), "Low price is inconsistent with OHLC"

print(f"\nMerged: {len(daily):,} trading sessions | {daily.index.min():%Y-%m-%d} ~ {daily.index.max():%Y-%m-%d}")
print(f"Missing/nonpositive volume: {(daily['volume'].isna() | (daily['volume'] <= 0)).sum():,} rows")
display(daily.head(3))

[1/5] Loading S&P 500 prices
  train: 6,037 rows | 1996-11-13 ~ 2020-11-06
  val  :   755 rows | 2020-11-09 ~ 2023-11-08
  test :   755 rows | 2023-11-09 ~ 2026-11-12

Merged: 7,547 trading sessions | 1996-11-13 ~ 2026-11-12
Missing/nonpositive volume: 1 rows


,timestamp,open,high,low,close,volume
signal_date,,,,,,
1996-11-13,847895400,729.56,732.11,728.03,731.13,429840000.0
1996-11-14,847981800,731.13,735.99,729.20,735.88,480350000.0
1996-11-15,848068200,735.88,741.92,735.15,737.62,529100000.0


## 2. Feature engineering: construct 14 features

| Experimental Setup | Number of Features | Purpose |
| :---: | :---: | :---: |
| Basic Features | 6 | Establish the baseline experiment |
| Extended Features | 14 | Verify whether the added features improve prediction performance |

1. Basic Features
- Base:      return_1d | return_5d | return_20d | volatility_20d | intraday_return | intraday_range
2. Extended Features
- Trend:     close_vs_sma20 | sma5_vs_sma20
- Intraday & Overnight: overnight_return | close_location
- Volatility: volatility_5d | avg_intraday_range_20d
- Volume:    volume_change_1d | volume_vs_prev20

In [7]:
# 2. Build 14 features using information through t
def build_features(bars: pd.DataFrame) -> pd.DataFrame:
    o, h, l, c = (bars[col] for col in ("open", "high", "low", "close"))
    v = bars["volume"].where(bars["volume"] > 0)
    price_range = h - l
    f = pd.DataFrame(index=bars.index)

    # Original 6: retain their names and formulas
    for n in (1, 5, 20):
        f[f"return_{n}d"] = c.pct_change(n, fill_method=None)
    f["volatility_20d"] = f["return_1d"].rolling(20).std(ddof=1)
    f["intraday_return"] = c / o - 1
    f["intraday_range"] = price_range / o

    # new features: trend (2)
    sma5, sma20 = c.rolling(5).mean(), c.rolling(20).mean()
    f["close_vs_sma20"] = c / sma20 - 1
    f["sma5_vs_sma20"] = sma5 / sma20 - 1

    # Overnight and intraday (2)
    f["overnight_return"] = o / c.shift(1) - 1
    f["close_location"] = (c - l) / price_range.where(price_range > 0)

    # Volatility (2)
    f["volatility_5d"] = f["return_1d"].rolling(5).std(ddof=1)
    f["avg_intraday_range_20d"] = f["intraday_range"].rolling(20).mean()

    # Volume (2): compare with the previous 20 days, excluding t
    f["volume_change_1d"] = v.pct_change(fill_method=None)
    f["volume_vs_prev20"] = v / v.shift(1).rolling(20).mean() - 1  # Compare today with the previous 20 days only
    return f

all_features = build_features(daily)
assert all_features.shape[1] == 14
assert not np.isinf(all_features.to_numpy(dtype=float)).any(), "Features contain infinity"

print("=" * 65)
print("[2/5] Feature construction complete")
print(f"Feature count: {all_features.shape[1]} (original 6 + new 8)")
print("Feature names: ", ", ".join(all_features.columns))
missing = all_features.isna().sum()
print("\nMissing features (nonzero counts only):")
print(missing[missing > 0].to_string() if (missing > 0).any() else "No missing values")
print(f"Complete feature rows: {all_features.notna().all(axis=1).sum():,}/{len(all_features):,}")
display(all_features.tail(3))

[2/5] Feature construction complete
Feature count: 14 (original 6 + new 8)
Feature names:  return_1d, return_5d, return_20d, volatility_20d, intraday_return, intraday_range, close_vs_sma20, sma5_vs_sma20, overnight_return, close_location, volatility_5d, avg_intraday_range_20d, volume_change_1d, volume_vs_prev20

Missing features (nonzero counts only):
return_1d                  1
return_5d                  5
return_20d                20
volatility_20d            20
close_vs_sma20            19
sma5_vs_sma20             19
overnight_return           1
volatility_5d              5
avg_intraday_range_20d    19
volume_change_1d           3
volume_vs_prev20          41
Complete feature rows: 7,506/7,547


,return_1d,return_5d,return_20d,volatility_20d,intraday_return,intraday_range,close_vs_sma20,sma5_vs_sma20,overnight_return,close_location,volatility_5d,avg_intraday_range_20d,volume_change_1d,volume_vs_prev20
signal_date,,,,,,,,,,,,,,
2026-11-10,-0.003842,-0.006340,-0.036451,0.013794,0.001044,0.009782,-0.014387,-0.017390,-0.004881,0.687168,0.026065,0.011823,-0.131116,-0.113807
2026-11-11,0.013090,-0.009006,-0.012898,0.013992,0.004314,0.009928,-0.000833,-0.018564,0.008738,0.630643,0.025651,0.011703,-0.111375,-0.213795
2026-11-12,-0.003541,0.032456,-0.019584,0.013977,0.005504,0.013498,-0.003380,-0.011321,-0.008995,0.407771,0.011728,0.011828,0.044476,-0.168186


## 3. Target Construction

| Task | Target |
| :---: | :---: |
| Regression | Predict actual trading return |
| Classification | Predict whether actual trading return exceeds 0.2% |

In [8]:
open_ = daily["open"]  # Use future opens solely to create targets
entry_open, exit_open = open_.shift(-1), open_.shift(-2)

# Three-decimal opens permit an exact integer test for > 0.002
open_mills = (open_ * 1000).round().astype("Int64")
entry_mills, exit_mills = open_mills.shift(-1), open_mills.shift(-2)

labels = pd.DataFrame({
    "trade_date": daily.index.to_series().shift(-1),
    "settle_date": daily.index.to_series().shift(-2),
    "target_return": (exit_open - entry_open) / entry_open,     # build regression labels
    "target_class": (500 * (exit_mills - entry_mills) > entry_mills).astype("Int8"),    # build classification labels
}).dropna()

features = all_features.loc[labels.index].copy()    # align features and labels

assert features.index.equals(labels.index), "Features and targets are misaligned"
assert labels.index.equals(daily.index[:-2]), "Final two sessions were not excluded"
assert labels["target_class"].isin([0, 1]).all()

count = labels["target_class"].value_counts().reindex([0, 1], fill_value=0)
print("=" * 65)
print("[3/5] Targets generated")
print(f"Valid samples: {len(labels):,} rows | signal dates {labels.index.min():%Y-%m-%d} ~ {labels.index.max():%Y-%m-%d}")
print(f"Class 0 (return <= +0.20%): {count[0]:,} ({count[0]/len(labels):.2%})")
print(f"Class 1 (return > +0.20%): {count[1]:,} ({count[1]/len(labels):.2%})")
print(f"Regression target: mean {labels['target_return'].mean():.4%} | median {labels['target_return'].median():.4%} | std {labels['target_return'].std():.4%}")
print("\nTarget examples:")
display(labels.head(5))

[3/5] Targets generated
Valid samples: 7,545 rows | signal dates 1996-11-13 ~ 2026-11-10
Class 0 (return <= +0.20%): 4,331 (57.40%)
Class 1 (return > +0.20%): 3,214 (42.60%)
Regression target: mean 0.0374% | median 0.0770% | std 1.1357%

Target examples:


,trade_date,settle_date,target_return,target_class
signal_date,,,,
1996-11-13,1996-11-14,1996-11-15,0.006497,1
1996-11-14,1996-11-15,1996-11-18,0.002365,1
1996-11-15,1996-11-18,1996-11-19,-0.000813,0
1996-11-18,1996-11-19,1996-11-20,0.006974,1
1996-11-19,1996-11-20,1996-11-21,0.002412,1


## 4. Data Leakage Check

> To verify the existence of data leakage, we randomly selected 6 predicted dates and additionally checked the dates near the boundary of the dataset partition. By deleting future data and recalculating features, as well as independently verifying predicted labels, the correctness of feature and label construction is checked.

In [9]:
random_dates = np.random.default_rng(42).choice(
    features.index.to_numpy(), size=min(6, len(features)), replace=False
)
boundary_dates = [daily.index[daily.index.get_loc(d) - 1] for d in split_starts]
check_dates = pd.DatetimeIndex(list(random_dates) + boundary_dates).unique().sort_values()

print("=" * 65)
print("[4/5] Timing and leakage checks")
for date in check_dates:
    truncated_features = build_features(daily.loc[:date])
    pd.testing.assert_series_equal(
        all_features.loc[date], truncated_features.loc[date], check_exact=True
    )
    pos = daily.index.get_loc(date)
    future_dates = daily.index[pos + 1:pos + 3]
    start, end = (Decimal(str(x)) for x in daily.loc[future_dates, "open"])
    expected_return = (end - start) / start
    row = labels.loc[date]
    assert [row["trade_date"], row["settle_date"]] == list(future_dates)
    assert np.isclose(row["target_return"], float(expected_return), rtol=0, atol=1e-14)
    assert row["target_class"] == int(expected_return > Decimal("0.002"))
    print(f"  PASS | {date:%Y-%m-%d} -> {future_dates[0]:%Y-%m-%d} -> {future_dates[1]:%Y-%m-%d} | class={row['target_class']}")

print(f"\nChecks passed!! \n{len(check_dates)} signal dates; truncated features and independent targets match.")

[4/5] Timing and leakage checks
  PASS | 1999-07-16 -> 1999-07-19 -> 1999-07-20 | class=0
  PASS | 2009-11-05 -> 2009-11-06 -> 2009-11-09 | class=1
  PASS | 2010-01-11 -> 2010-01-12 -> 2010-01-13 | class=0
  PASS | 2016-06-27 -> 2016-06-28 -> 2016-06-29 | class=1
  PASS | 2020-01-24 -> 2020-01-27 -> 2020-01-28 | class=1
  PASS | 2020-11-06 -> 2020-11-09 -> 2020-11-10 | class=0
  PASS | 2022-08-11 -> 2022-08-12 -> 2022-08-15 | class=1
  PASS | 2023-11-08 -> 2023-11-09 -> 2023-11-10 | class=0

Checks passed!! 
8 signal dates; truncated features and independent targets match.


## 5. Export analysis feature table

In [10]:
# 5. Export three CSVs and verify reads
feature_notes = pd.DataFrame([
    ("return_1d", "Original | Trend", "C[t] / C[t-1] - 1"),
    ("return_5d", "Original | Trend", "C[t] / C[t-5] - 1"),
    ("return_20d", "Original | Trend", "C[t] / C[t-20] - 1"),
    ("volatility_20d", "Original | Volatility", "Sample std of last 20 daily returns"),
    ("intraday_return", "Original | Overnight/intraday", "C[t] / O[t] - 1"),
    ("intraday_range", "Original | Overnight/intraday", "(H[t] - L[t]) / O[t]"),
    ("close_vs_sma20", "Trend", "C[t] / SMA20[t] - 1"),
    ("sma5_vs_sma20", "Trend", "SMA5[t] / SMA20[t] - 1"),
    ("overnight_return", "Overnight/intraday", "O[t] / C[t-1] - 1"),
    ("close_location", "Overnight/intraday", "(C[t] - L[t]) / (H[t] - L[t]); missing when range is zero"),
    ("volatility_5d", "Volatility", "Sample std of last 5 daily returns"),
    ("avg_intraday_range_20d", "Volatility", "Mean of last 20 intraday ranges"),
    ("volume_change_1d", "Volume", "V[t] / V[t-1] - 1"),
    ("volume_vs_prev20", "Volume", "V[t] / mean(V[t-20], …, V[t-1]) - 1"),  # Compare today with the previous 20 days only
], columns=["feature", "group", "formula"])
assert feature_notes["feature"].tolist() == features.columns.tolist()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "sp500_features.csv": features,
    "sp500_labels.csv": labels,
    "sp500_feature_notes.csv": feature_notes,
}
print("=" * 65)
print("[5/5] Export complete")
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=(filename != "sp500_feature_notes.csv"),
                 encoding="utf-8-sig", date_format="%Y-%m-%d")
    assert len(pd.read_csv(path)) == len(table), f"Read-back row count mismatch: {filename}"  # Check row counts after saving
    print(f"  OK | {filename:26s} | {len(table):5,d} rows | {path.stat().st_size / 1024:.1f} KB")

print(f"\nOutput directory: {OUTPUT_DIR.relative_to(ROOT)}")

[5/5] Export complete
  OK | sp500_features.csv         | 7,545 rows | 2200.8 KB
  OK | sp500_labels.csv           | 7,545 rows | 424.8 KB
  OK | sp500_feature_notes.csv    |    14 rows | 0.8 KB

Output directory: data\features


## 6. Prepare model inputs

Classification A/B share dates from 2009 (6 versus 11 features). Both regression models share 6 features.

In [11]:
# Do not drop six-feature samples because volume is missing in the full feature table.
original = all_features[ORIGINAL]
market, audit = load_market_features(daily.index)
# build dataset for 2 tasks
regression_samples = add_targets(daily, original)
classification_samples = add_targets(daily, original.join(market))
classification_samples = classification_samples.loc["2009-01-01":].dropna(subset=ORIGINAL + MARKET)
save_csv(OUTPUT_DIR / "market_availability.csv", audit, index=False)

for task, samples in (("classification", classification_samples), ("regression", regression_samples)):
    save_csv(OUTPUT_DIR / f"{task}_samples.csv", samples)
    splits, summary = chronological_split(samples)  # Drop targets crossing split boundaries
    split_dir = ROOT / "data" / "model_splits" / task
    split_dir.mkdir(parents=True, exist_ok=True)
    for name, frame in splits.items():
        save_csv(split_dir / f"{name}.csv", frame)
    save_csv(split_dir / "split_summary.csv", summary, index=False)
    print(f"\n{task} valid samples: {len(samples):,}")
    display(summary)

# Hash inputs and outputs to detect stale features without machine-specific paths.
sources = [RAW_FILE, CLEAN_FILE, *FILES.values(), *MARKET_FILES]


classification valid samples: 4,492


,split,samples,first_signal,last_signal,first_entry,last_settlement
0,train,3593,2009-01-02,2023-04-12,2009-01-05,2023-04-14
1,val,448,2023-04-14,2025-01-27,2023-04-17,2025-01-29
2,test,449,2025-01-29,2026-11-10,2025-01-30,2026-11-12



regression valid samples: 7,525


,split,samples,first_signal,last_signal,first_entry,last_settlement
0,train,6020,1996-12-12,2020-11-11,1996-12-13,2020-11-13
1,val,751,2020-11-13,2023-11-08,2020-11-16,2023-11-10
2,test,752,2023-11-10,2026-11-10,2023-11-13,2026-11-12
